# Three private sugar companies

Balrampur Chini, Triveni Engineering and Bajaj Hindusthan Sugar all crush cane in UP, so they all pay the same state cane price. That makes them a fair comparison for the main question here, whether ethanol helped mills pay farmers faster.

None of the three show cane dues as a separate balance sheet line, the dues sit inside trade payables. So I use payable days, trade payables at 31 March divided by the year's cost of materials consumed (mostly cane), times 365. A mill that pays farmers slowly shows a lot more payable days.

All figures are standalone and in lakh rupees. I typed them from each company's annual report into `firm_typed.csv`, and `firm_sources.csv` has the link to every report. The page columns are PDF page numbers so anyone can check a figure.

In [1]:
import pandas as pd

RAW = "data/raw/"

firms = pd.read_csv(RAW + "firm_typed.csv")
sources = pd.read_csv(RAW + "firm_sources.csv")

# every row should point to a report in the source list
missing = set(firms["report"]) - set(sources["report"])
missing

set()

Distillery share uses the sugar and distillery segment revenues only, so engineering and other businesses don't dilute it. One thing to keep in mind, Triveni also sells potable liquor, so its distillery revenue includes excise duty and its share runs a bit high.

In [2]:
firms["payable_days"] = (firms["trade_payables"] / firms["cogs"] * 365).round(1)
firms["distillery_share"] = (firms["distillery_rev"] / (firms["distillery_rev"] + firms["sugar_rev"]) * 100).round(1)
firms["debt"] = firms["borrow_noncurrent"] + firms["borrow_current"]

firms.pivot(index="fy", columns="company", values="payable_days")

company,Bajaj Hindusthan,Balrampur,Triveni
fy,,,
2016-17,251.8,41.3,45.4
2017-18,251.2,92.9,88.8
2018-19,268.5,61.5,84.6
2019-20,302.1,70.6,91.7
2020-21,307.3,61.6,70.9
2021-22,324.0,31.1,41.1
2022-23,324.3,33.7,38.3
2023-24,264.1,22.2,32.8
2024-25,291.9,24.6,47.1


In [3]:
firms.pivot(index="fy", columns="company", values="distillery_share")

company,Bajaj Hindusthan,Balrampur,Triveni
fy,,,
2016-17,7.6,7.9,5.7
2017-18,6.2,6.6,3.5
2018-19,5.9,9.5,7.3
2019-20,3.3,10.0,8.8
2020-21,4.8,16.2,11.8
2021-22,13.0,19.0,23.6
2022-23,12.5,21.2,30.0
2023-24,10.7,26.4,36.4
2024-25,9.4,22.6,39.5


A first look at whether the two move together. Inside each company I check the correlation between distillery share and payable days across the ten years. With ten points per company this is only a description, not proof, and other things changed in the same years (sugar prices, the minimum selling price, Bajaj's debt restructuring).

In [4]:
firms.groupby("company")[["distillery_share", "payable_days"]].corr().xs("payable_days", level=1)["distillery_share"].round(2)

company
Bajaj Hindusthan    0.35
Balrampur          -0.84
Triveni            -0.79
Name: distillery_share, dtype: float64

In [5]:
cols = ["company", "fy", "payable_days", "distillery_share", "debt", "finance_costs", "trade_payables", "cogs", "sugar_rev", "distillery_rev"]
firms[cols].to_csv("data/clean/firms.csv", index=False)
firms[cols].head()

,company,fy,payable_days,distillery_share,debt,finance_costs,trade_payables,cogs,sugar_rev,distillery_rev
0,Balrampur,2025-26,22.4,23.8,316935.41,7722.45,28230.60,460885.05,550722.98,172096.97
1,Balrampur,2024-25,24.6,22.6,262619.15,9346.09,27664.71,411135.33,489741.47,143001.24
2,Balrampur,2023-24,22.2,26.4,200825.30,8362.62,27843.81,457983.28,469731.38,168901.19
3,Balrampur,2022-23,33.7,21.2,187862.05,4864.68,31601.90,341939.67,433857.45,116397.55
4,Balrampur,2021-22,31.1,19.0,120962.90,3086.89,27784.55,326173.02,426298.12,100141.09
